In [1]:
# =========================================================
# TASK 5
# TRACK BEST VALIDATION ACCURACY USING W&B
# COMPLETE WORKING CODE
# =========================================================


# =========================
# INSTALL W&B
# =========================

# Run only once in Colab/Jupyter
# !pip install wandb


# =========================
# IMPORT LIBRARIES
# =========================

import numpy as np
import wandb

from tensorflow.keras.datasets import fashion_mnist
from sklearn.model_selection import train_test_split


# =========================
# LOGIN TO W&B
# =========================

wandb.login()


# =========================
# LOAD DATASET
# =========================

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()


# =========================
# PREPROCESS DATA
# =========================

# Flatten images
x_train = x_train.reshape(x_train.shape[0], 784) / 255.0
x_test = x_test.reshape(x_test.shape[0], 784) / 255.0


# =========================
# VALIDATION SPLIT
# =========================

x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.1,
    random_state=42
)


# =========================
# ONE HOT ENCODING
# =========================

def one_hot_encode(y, num_classes=10):

    one_hot = np.zeros((y.size, num_classes))

    one_hot[np.arange(y.size), y] = 1

    return one_hot


y_train_encoded = one_hot_encode(y_train)
y_val_encoded = one_hot_encode(y_val)


# =========================
# ACTIVATION FUNCTIONS
# =========================

def sigmoid(z):

    return 1 / (1 + np.exp(-z))


def sigmoid_derivative(z):

    s = sigmoid(z)

    return s * (1 - s)


def tanh(z):

    return np.tanh(z)


def tanh_derivative(z):

    return 1 - np.tanh(z) ** 2


def relu(z):

    return np.maximum(0, z)


def relu_derivative(z):

    return (z > 0).astype(float)


def softmax(z):

    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))

    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


# =========================
# ACTIVATION SELECTOR
# =========================

def get_activation(name):

    if name == "sigmoid":

        return sigmoid, sigmoid_derivative

    elif name == "tanh":

        return tanh, tanh_derivative

    elif name == "relu":

        return relu, relu_derivative

    else:

        raise ValueError("Invalid activation")


# =========================
# LOSS FUNCTION
# =========================

def cross_entropy_loss(y_true, y_pred):

    epsilon = 1e-10

    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

    return loss


# =========================
# ACCURACY FUNCTION
# =========================

def accuracy(y_true, y_pred):

    predictions = np.argmax(y_pred, axis=1)

    return np.mean(predictions == y_true)


# =========================
# FEEDFORWARD NEURAL NETWORK
# =========================

class FeedForwardNeuralNetwork:

    def __init__(
        self,
        input_size,
        hidden_layers,
        output_size,
        activation_name="relu"
    ):

        self.layers = [input_size] + hidden_layers + [output_size]

        self.weights = []
        self.biases = []

        self.activation_name = activation_name

        self.activation, self.activation_derivative = (
            get_activation(activation_name)
        )

        # Xavier Initialization
        for i in range(len(self.layers) - 1):

            weight = np.random.randn(
                self.layers[i],
                self.layers[i + 1]
            ) * np.sqrt(1 / self.layers[i])

            bias = np.zeros((1, self.layers[i + 1]))

            self.weights.append(weight)
            self.biases.append(bias)

        # Adam Variables
        self.m_w = [np.zeros_like(w) for w in self.weights]
        self.m_b = [np.zeros_like(b) for b in self.biases]

        self.v_w = [np.zeros_like(w) for w in self.weights]
        self.v_b = [np.zeros_like(b) for b in self.biases]

    # =========================
    # FORWARD PROPAGATION
    # =========================

    def forward(self, X):

        activations = [X]
        z_values = []

        A = X

        for i in range(len(self.weights) - 1):

            Z = np.dot(A, self.weights[i]) + self.biases[i]

            z_values.append(Z)

            A = self.activation(Z)

            activations.append(A)

        Z = np.dot(A, self.weights[-1]) + self.biases[-1]

        z_values.append(Z)

        output = softmax(Z)

        activations.append(output)

        return activations, z_values

    # =========================
    # BACKPROPAGATION
    # =========================

    def backward(self, X, y, activations, z_values):

        m = X.shape[0]

        gradients_w = []
        gradients_b = []

        dZ = activations[-1] - y

        for i in reversed(range(len(self.weights))):

            dW = np.dot(activations[i].T, dZ) / m

            dB = np.sum(dZ, axis=0, keepdims=True) / m

            gradients_w.insert(0, dW)
            gradients_b.insert(0, dB)

            if i > 0:

                dA = np.dot(dZ, self.weights[i].T)

                dZ = (
                    dA *
                    self.activation_derivative(
                        z_values[i - 1]
                    )
                )

        return gradients_w, gradients_b

    # =========================
    # SGD OPTIMIZER
    # =========================

    def sgd(
        self,
        gradients_w,
        gradients_b,
        learning_rate
    ):

        for i in range(len(self.weights)):

            self.weights[i] -= (
                learning_rate * gradients_w[i]
            )

            self.biases[i] -= (
                learning_rate * gradients_b[i]
            )

    # =========================
    # ADAM OPTIMIZER
    # =========================

    def adam(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        t,
        beta1=0.9,
        beta2=0.999,
        epsilon=1e-8
    ):

        for i in range(len(self.weights)):

            self.m_w[i] = (
                beta1 * self.m_w[i]
                + (1 - beta1) * gradients_w[i]
            )

            self.m_b[i] = (
                beta1 * self.m_b[i]
                + (1 - beta1) * gradients_b[i]
            )

            self.v_w[i] = (
                beta2 * self.v_w[i]
                + (1 - beta2) * (gradients_w[i] ** 2)
            )

            self.v_b[i] = (
                beta2 * self.v_b[i]
                + (1 - beta2) * (gradients_b[i] ** 2)
            )

            m_w_hat = self.m_w[i] / (
                1 - beta1 ** t
            )

            m_b_hat = self.m_b[i] / (
                1 - beta1 ** t
            )

            v_w_hat = self.v_w[i] / (
                1 - beta2 ** t
            )

            v_b_hat = self.v_b[i] / (
                1 - beta2 ** t
            )

            self.weights[i] -= (
                learning_rate *
                m_w_hat /
                (np.sqrt(v_w_hat) + epsilon)
            )

            self.biases[i] -= (
                learning_rate *
                m_b_hat /
                (np.sqrt(v_b_hat) + epsilon)
            )

    # =========================
    # UPDATE PARAMETERS
    # =========================

    def update_parameters(
        self,
        gradients_w,
        gradients_b,
        learning_rate,
        optimizer="sgd",
        t=1
    ):

        if optimizer == "adam":

            self.adam(
                gradients_w,
                gradients_b,
                learning_rate,
                t
            )

        else:

            self.sgd(
                gradients_w,
                gradients_b,
                learning_rate
            )


# =========================
# TRAIN FUNCTION
# =========================

def train():

    wandb.init()

    config = wandb.config

    # Run Name
    wandb.run.name = (
        f"hl_{config.num_hidden_layers}"
        f"_bs_{config.batch_size}"
        f"_ac_{config.activation}"
    )

    # Create Hidden Layers
    hidden_layers = [
        config.hidden_size
        for _ in range(config.num_hidden_layers)
    ]

    # Create Model
    model = FeedForwardNeuralNetwork(
        input_size=784,
        hidden_layers=hidden_layers,
        output_size=10,
        activation_name=config.activation
    )

    best_val_accuracy = 0

    # Training Loop
    for epoch in range(config.epochs):

        # Forward Pass
        activations, z_values = model.forward(x_train)

        # Training Loss
        train_loss = cross_entropy_loss(
            y_train_encoded,
            activations[-1]
        )

        # Backpropagation
        gradients_w, gradients_b = model.backward(
            x_train,
            y_train_encoded,
            activations,
            z_values
        )

        # Parameter Update
        model.update_parameters(
            gradients_w,
            gradients_b,
            learning_rate=config.learning_rate,
            optimizer=config.optimizer,
            t=epoch + 1
        )

        # Validation
        val_activations, _ = model.forward(x_val)

        val_loss = cross_entropy_loss(
            y_val_encoded,
            val_activations[-1]
        )

        val_accuracy = accuracy(
            y_val,
            val_activations[-1]
        )

        # Store Best Accuracy
        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy

        # Log Metrics
        wandb.log({

            "epoch": epoch + 1,

            "train_loss": train_loss,

            "val_loss": val_loss,

            "val_accuracy": val_accuracy,

            "best_val_accuracy": best_val_accuracy
        })

    print("Best Validation Accuracy:",
          best_val_accuracy)


# =========================
# SWEEP CONFIGURATION
# =========================

sweep_config = {

    "method": "bayes",

    "metric": {

        "name": "best_val_accuracy",

        "goal": "maximize"
    },

    "parameters": {

        "epochs": {

            "values": [5, 10]
        },

        "num_hidden_layers": {

            "values": [3, 4]
        },

        "hidden_size": {

            "values": [32, 64, 128]
        },

        "learning_rate": {

            "values": [1e-3, 1e-4]
        },

        "optimizer": {

            "values": [
                "sgd",
                "adam"
            ]
        },

        "batch_size": {

            "values": [16, 32]
        },

        "activation": {

            "values": [
                "relu",
                "tanh",
                "sigmoid"
            ]
        }
    }
}


# =========================
# CREATE SWEEP
# =========================

sweep_id = wandb.sweep(

    sweep_config,

    project="fashion-mnist-assignment"
)


# =========================
# RUN SWEEP AGENT
# =========================

wandb.agent(

    sweep_id,

    function=train,

    count=10
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: prabhaa-aids2023 (vaishalinir-ymc2022-chennai-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

Best Validation Accuracy: 0.5885


best_val_accuracy,▁▂▄▆▇▇▇███
epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▇▆▅▄▃▂▂▁▁
val_accuracy,▁▂▄▆▇▇▇███
val_loss,█▇▆▅▄▃▃▂▁▁
best_val_accuracy,0.5885
epoch,10
train_loss,1.38013
val_accuracy,0.5885
val_loss,1.34199


wandb: Agent Starting Run: h59v10zj with config:
wandb: 	activation: tanh
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.0001
wandb: 	num_hidden_layers: 3
wandb: 	optimizer: adam
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Best Validation Accuracy: 0.4555


best_val_accuracy,▁▁▂▂▃▄▅▆▇█
epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▇▆▅▄▄▃▂▂▁
val_accuracy,▁▁▂▂▃▄▅▆▇█
val_loss,█▇▆▅▅▄▃▂▂▁
best_val_accuracy,0.4555
epoch,10
train_loss,2.05721
val_accuracy,0.4555
val_loss,2.03051


# Task 5: Best Validation Accuracy Across All Models

## Objective
The objective of this task is to identify and analyze the best validation accuracy achieved across all neural network models trained during the hyperparameter sweep experiments.

Weights & Biases (WandB) is used to automatically track, compare, and visualize the performance of different models trained with varying hyperparameter configurations.

---

# Approach

A hyperparameter sweep was conducted using WandB to explore multiple combinations of:
- number of hidden layers
- hidden layer size
- activation functions
- optimizers
- learning rates
- batch sizes
- number of epochs

Each experiment trained a feedforward neural network implemented entirely from scratch using NumPy.

The validation accuracy for every model was recorded after each training epoch.

---

# Validation Accuracy Tracking

During training, the following metrics were logged to WandB:
- training loss
- validation loss
- validation accuracy
- best validation accuracy

The best validation accuracy was updated whenever the current validation accuracy exceeded the previous best value.

Example:
```python
if val_accuracy > best_val_accuracy:
    best_val_accuracy = val_accuracy